In [50]:
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq                          
import os

llm = ChatGroq(model="openai/gpt-oss-120b",temperature=0)

In [51]:
from langchain.tools import tool

In [52]:
import requests
response=requests.get("https://api.exchangerate-api.com/v4/latest/EUR")
data=response.json()
print(data)

{'provider': 'https://www.exchangerate-api.com', 'WARNING_UPGRADE_TO_V6': 'https://www.exchangerate-api.com/docs/free', 'terms': 'https://www.exchangerate-api.com/terms', 'base': 'EUR', 'date': '2026-05-20', 'time_last_updated': 1779235201, 'rates': {'EUR': 1, 'AED': 4.26, 'AFN': 73.49, 'ALL': 95.47, 'AMD': 427.82, 'ANG': 2.08, 'AOA': 1095.82, 'ARS': 1624.38, 'AUD': 1.63, 'AWG': 2.08, 'AZN': 1.98, 'BAM': 1.96, 'BBD': 2.32, 'BDT': 142.63, 'BGN': 1.96, 'BHD': 0.437, 'BIF': 3474.09, 'BMD': 1.16, 'BND': 1.49, 'BOB': 8.06, 'BRL': 5.84, 'BSD': 1.16, 'BTN': 112.09, 'BWP': 16.38, 'BYN': 3.16, 'BZD': 2.32, 'CAD': 1.6, 'CDF': 2665.38, 'CHF': 0.916, 'CLF': 0.0265, 'CLP': 1047.09, 'CNH': 7.91, 'CNY': 7.92, 'COP': 4412.94, 'CRC': 527.44, 'CUP': 27.87, 'CVE': 110.27, 'CZK': 24.31, 'DJF': 206.36, 'DKK': 7.46, 'DOP': 68.72, 'DZD': 154.28, 'EGP': 61.67, 'ERN': 17.42, 'ETB': 184.34, 'FJD': 2.56, 'FKP': 0.867, 'FOK': 7.46, 'GBP': 0.867, 'GEL': 3.11, 'GGP': 0.867, 'GHS': 13.34, 'GIP': 0.867, 'GMD': 86.26,

In [53]:
@tool
def tool_euroconversion(amount : float) -> str :
    """ Use this tool to convert EUR to INR.
    Extract ONLY the numeric value from the user's message and pass it as amount.
    For example if user says 'what is 100 EUR in INR', pass amount=100.0
    IMPORTANT: Always use the exact number returned by this tool in your answer. 
    Never use your own knowledge for exchange rates rely on the value of the tool. """

    import requests
    response=requests.get("https://api.exchangerate-api.com/v4/latest/EUR")
    data=response.json()
    rate=data["rates"]["INR"]
    result=amount*rate

    return f"The live exchange rate fetched right now is: 1 EUR = {rate} INR. Therefore {amount} EUR = {result} INR. This is a real time result, do not modify it dont use your own exchange rate it's incorrect rely on the value of the tool."


In [54]:
tool_euroconversion.invoke({"amount": 1})

"The live exchange rate fetched right now is: 1 EUR = 112.09 INR. Therefore 1.0 EUR = 112.09 INR. This is a real time result, do not modify it dont use your own exchange rate it's incorrect rely on the value of the tool."

In [55]:
@tool
def tool_personal_info(name: str) -> str:
    """Use this tool when you need to answer questions about personal information.
    Args:
        name (str): The name of the person to look up.
    Returns:
        str: A string containing the person's age and occupation, or a message if the information is not found.
    """
    
    infos = [{
        "name": "Dhruv Patel",
        "age": 25,
        "occupation": "Data analyst"
    },
    {
        "name": "Jane Smith",
        "age": 25,
        "occupation": "Data Scientist"
    }]

    for info in infos:
        if info["name"].lower() == name.lower():
            return f"{info['name']} is {info['age']} years old and works as a {info['occupation']}."
    return "Information not found."

In [56]:
tool_personal_info.invoke("Dhruv Patel")

'Dhruv Patel is 25 years old and works as a Data analyst.'

In [57]:
toolkit=[
    tool_euroconversion,
    tool_personal_info
]

In [58]:
llm_bind=llm.bind_tools(toolkit)

In [59]:
llm_bind.invoke("what is the price of 100 EUR in INR?")

AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks: "what is the price of 100 EUR in INR?" Need to use tool_euroconversion with amount=100.', 'tool_calls': [{'id': 'fc_6c1681b1-2e1a-4c4f-8437-9c3c6dcc8ff5', 'function': {'arguments': '{"amount":100}', 'name': 'tool_euroconversion'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 281, 'total_tokens': 338, 'completion_time': 0.121066918, 'completion_tokens_details': {'reasoning_tokens': 28}, 'prompt_time': 0.013495896, 'prompt_tokens_details': None, 'queue_time': 0.018257578, 'total_time': 0.134562814}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_87b3c396db', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e4553-9c60-7af2-b39e-9c6cd856107a-0', tool_calls=[{'name': 'tool_euroconversion', 'args': {'amount': 100}, 'id': 'fc_6c1681b1-2e1a-4c4f-8437-9c3c6dcc8ff5', 'type': 'tool_ca

In [60]:
from langchain.agents import create_agent

agent=create_agent(llm_bind,toolkit)

In [62]:
agent.invoke(
    {'messages':[{"role" : "user", "content" : " what is the price of 2 EUR in INR use the tools value? what is the age of dhruv patel just the age ?"}]}
)

{'messages': [HumanMessage(content=' what is the price of 2 EUR in INR use the tools value? what is the age of dhruv patel just the age ?', additional_kwargs={}, response_metadata={}, id='8bd91348-8d9f-470a-9762-2e809dbd3957'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to answer two queries: price of 2 EUR in INR using the tool, and age of Dhruv Patel using personal info tool. Use tool_euroconversion with amount 2. Then tool_personal_info with name "Dhruv Patel". We\'ll call both.', 'tool_calls': [{'id': 'fc_98dd8739-ed4a-45cb-8c1c-cc33b64091eb', 'function': {'arguments': '{"amount":2}', 'name': 'tool_euroconversion'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 87, 'prompt_tokens': 299, 'total_tokens': 386, 'completion_time': 0.190161582, 'completion_tokens_details': {'reasoning_tokens': 58}, 'prompt_time': 0.158602121, 'prompt_tokens_details': {'cached_tokens': 256}, 'queue_time': 0.017145992, 'total_time': 0.348763703